In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, Add, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

In [2]:

# Load the dataset
data = pd.read_csv('imbalance_data_CIC-IDS-2017 Dataset.csv')

# Replace infinite values with NaN and fill NaN values with column mean
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.fillna(data.mean(), inplace=True)

# Define the dependent and independent variables
X = data.drop(['Unnamed: 0', 'label'], axis=1)  # Independent variables
y = data['label']  # Dependent variable


In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
# Reshape the data for ResNet (assuming features are 1D and need to be reshaped to 2D)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1, 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1, 1)

In [5]:
# Define ResNet block
def resnet_block(input_tensor, filters, kernel_size=3):
    x = Conv2D(filters, kernel_size, padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, input_tensor])
    x = Activation('relu')(x)
    return x

In [6]:
# Build ResNet model
input_shape = (X_train.shape[1], X_train.shape[2], X_train.shape[3])
inputs = Input(shape=input_shape)
x = resnet_block(inputs, filters=64)
x = resnet_block(x, filters=64)
x = resnet_block(x, filters=64)
x = GlobalAveragePooling2D()(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)


In [7]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [8]:
# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

# Print the classification report and confusion matrix
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Epoch 1/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 1155s 16ms/step - accuracy: 0.4069 - loss: -73131.0547 - val_accuracy: 0.2161 - val_loss: -251380.1875
Epoch 2/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 1203s 17ms/step - accuracy: 0.4188 - loss: -1002171.9375 - val_accuracy: 0.0606 - val_loss: -1868538.6250
Epoch 3/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 1350s 19ms/step - accuracy: 0.4216 - loss: -3160969.0000 - val_accuracy: 0.0923 - val_loss: -3205125.2500
Epoch 4/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 1521s 22ms/step - accuracy: 0.4245 - loss: -6568516.0000 - val_accuracy: 0.1910 - val_loss: -5669453.5000
Epoch 5/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 1747s 25ms/step - accuracy: 0.4294 - loss: -11225231.0000 - val_accuracy: 0.5910 - val_loss: -1986566.2500
Epoch 6/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 2044s 29ms/step - accuracy: 0.4302 - loss: -17249040.0000 - val_accuracy: 0.3134 - val_loss: -9618583.0000
Epoch 7/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 2431s 34ms/step - accuracy: 0.4335 - loss: -24577534.0000

C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

           0       0.99      0.70      0.82    454556
           1       0.00      0.80      0.00       404
           2       0.00      0.00      0.00     25591
           3       0.00      0.00      0.00      2077
           4       0.00      0.00      0.00     46051
           5       0.00      0.00      0.00      1061
           6       0.00      0.00      0.00      1155
           7       0.00      0.00      0.00      1631
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         3
          10       0.00      0.00      0.00     31692
          11       0.00      0.00      0.00      1171
          12       0.00      0.00      0.00       312
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00       129

    accuracy                           0.56    565836
   macro avg       0.07      0.10      0.05    565836
weighted avg       0.80   

C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
